# Trích ảnh từ video .MOV

Hàm xuất **80–150 ảnh**, thử nhiều số lượng để tìm overlap ước lượng gần **70–80%** nhất. Overlap được ước lượng theo vùng ảnh chung từ ORB + homography; chuyển động camera, thị sai, vật thể chuyển động hoặc vùng ít chi tiết có thể làm ước lượng sai/không khả dụng. Không thể bảo đảm overlap thực tế chỉ từ FPS.

Hàm giữ độ phân giải ảnh gốc, ghi `frames.csv` để kiểm tra và chỉ ghi vào thư mục mới/rỗng. Chạy cell cài thư viện nếu môi trường còn thiếu.


In [1]:
from pathlib import Path
import csv
import warnings

import cv2
import numpy as np


In [2]:
def video_to_frames(video_path, output_dir, min_images=80, max_images=150,
                    target_overlap=0.75, overlap_range=(0.70, 0.80),
                    count_step=10, analysis_width=640, jpeg_quality=95,
                    start_time=0.0, end_time=None):
    """Extract MOV frames; choose image count using estimated visual overlap.

    Overlap is a homography-based shared image-area estimate, not a guarantee
    of true 3D scene overlap. Works best with slow motion and textured scenes.
    Output directory must be new or empty; original video is never modified.
    Requires: opencv-python, numpy.
    """
    if not (80 <= min_images <= max_images <= 150):
        raise ValueError("Require 80 <= min_images <= max_images <= 150.")
    lo, hi = overlap_range
    if not (0.70 <= lo <= target_overlap <= hi <= 0.80):
        raise ValueError("Overlap range/target must be within 0.70–0.80.")
    if count_step < 1 or analysis_width < 64 or not 1 <= jpeg_quality <= 100:
        raise ValueError("Invalid count_step, analysis_width or jpeg_quality.")
    video_path, output_dir = Path(video_path), Path(output_dir)
    if not video_path.is_file():
        raise FileNotFoundError(video_path)
    if output_dir.exists() and any(output_dir.iterdir()):
        raise FileExistsError(f"Choose a new/empty output directory: {output_dir}")

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        cap.release()
        raise RuntimeError("Cannot open video. Check MOV codec support in OpenCV.")
    orb = cv2.ORB_create(nfeatures=1800)
    matcher = cv2.BFMatcher(cv2.NORM_HAMMING)
    features, pair_cache = {}, {}

    def read_frame(index):
        if not cap.set(cv2.CAP_PROP_POS_FRAMES, int(index)):
            raise RuntimeError(f"Cannot seek to frame {index}.")
        ok, frame = cap.read()
        if not ok:
            raise RuntimeError(f"Cannot decode frame {index}.")
        return frame

    def feature(index):
        index = int(index)
        if index not in features:
            frame = read_frame(index)
            h, w = frame.shape[:2]
            scale = min(1.0, analysis_width / w)
            gray = cv2.cvtColor(cv2.resize(
                frame, (round(w * scale), round(h * scale))), cv2.COLOR_BGR2GRAY)
            keypoints, descriptors = orb.detectAndCompute(gray, None)
            features[index] = (keypoints, descriptors, gray.shape)
        return features[index]

    def overlap(a, b):
        key = (int(a), int(b))
        if key in pair_cache:
            return pair_cache[key]
        ka, da, sa = feature(a)
        kb, db, sb = feature(b)
        value = float("nan")
        if da is not None and db is not None and len(da) >= 12 and len(db) >= 12:
            matches = matcher.knnMatch(da, db, k=2)
            good = [m[0] for m in matches
                    if len(m) == 2 and m[0].distance < 0.75 * m[1].distance]
            if len(good) >= 12:
                pa = np.float32([ka[m.queryIdx].pt for m in good])
                pb = np.float32([kb[m.trainIdx].pt for m in good])
                H, mask = cv2.findHomography(pa, pb, cv2.RANSAC, 3.0)
                if H is not None and mask is not None and mask.sum() >= 12 and mask.mean() >= 0.5:
                    ha, wa = sa
                    hb, wb = sb
                    ca = np.float32([[0, 0], [wa, 0], [wa, ha], [0, ha]])
                    cb = np.float32([[0, 0], [wb, 0], [wb, hb], [0, hb]])
                    projected = cv2.perspectiveTransform(ca[None], H)[0]
                    if np.isfinite(projected).all() and cv2.isContourConvex(projected):
                        area_a = abs(cv2.contourArea(projected))
                        area_b = float(wb * hb)
                        if 0.25 * area_b <= area_a <= 4 * area_b:
                            intersection, _ = cv2.intersectConvexConvex(projected, cb)
                            # Conservative shared area relative to both views.
                            value = float(np.clip(intersection / max(area_a, area_b), 0, 1))
        pair_cache[key] = value
        return value

    try:
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = float(cap.get(cv2.CAP_PROP_FPS))

        if not np.isfinite(fps) or fps <= 0 or total <= 0:
            raise ValueError("Video phải có FPS và số khung hình hợp lệ.")

        duration = total / fps
        start_time = float(start_time)
        end_time = duration if end_time is None else float(end_time)

        if not np.isfinite(start_time) or not np.isfinite(end_time):
            raise ValueError("Thời gian phải là số hữu hạn.")

        if not (0 <= start_time < end_time <= duration):
            raise ValueError(
                f"Cần 0 <= start_time < end_time <= {duration:.6f} giây."
            )

        # Bao gồm mốc bắt đầu, loại trừ mốc kết thúc.
        first_frame = int(np.ceil(start_time * fps))
        stop_frame = min(total, int(np.ceil(end_time * fps)))
        segment_frames = stop_frame - first_frame

        if segment_frames < min_images:
            raise ValueError(
                f"Đoạn đã chọn chỉ có {segment_frames} khung hình; "
                f"cần ít nhất {min_images}."
            )

        upper = min(max_images, segment_frames)
        counts = sorted(set(range(min_images, upper + 1, count_step)) | {upper})
        best = None
        print(f"Analyzing {start_time:.3f}–{end_time:.3f}s "f"({segment_frames} frames); trying image counts: {counts}")
        for count in counts:
            indices = np.rint(np.linspace(first_frame, stop_frame - 1, count)).astype(int)
            estimates = np.array([overlap(a, b) for a, b in zip(indices[:-1], indices[1:])])
            valid = np.isfinite(estimates)
            in_range = valid & (estimates >= lo) & (estimates <= hi)
            coverage = float(in_range.mean())
            error = float(np.abs(estimates[valid] - target_overlap).mean()) if valid.any() else float("inf")
            score = (coverage, float(valid.mean()), -error, -abs(count - 120))
            if best is None or score > best[0]:
                best = (score, indices, estimates)

        _, indices, estimates = best
        output_dir.mkdir(parents=True, exist_ok=True)
        rows, paths = [], []
        for number, index in enumerate(indices, 1):
            frame = read_frame(index)
            path = output_dir / f"frame_{number:04d}.jpg"
            ok, encoded = cv2.imencode(".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, jpeg_quality])
            if not ok:
                raise RuntimeError(f"Failed to encode frame {index}.")
            encoded.tofile(str(path))  # Supports Windows Unicode paths.
            paths.append(path)
            previous = estimates[number - 2] if number > 1 else float("nan")
            rows.append({
                "image": path.name, "source_frame": int(index),
                "seconds": round(int(index) / fps, 6) if np.isfinite(fps) and fps > 0 else "",
                "overlap_previous": float(previous) if np.isfinite(previous) else "",
            })
        with (output_dir / "frames.csv").open("w", newline="", encoding="utf-8-sig") as file:
            writer = csv.DictWriter(file, fieldnames=rows[0].keys())
            writer.writeheader()
            writer.writerows(rows)

        valid = np.isfinite(estimates)
        in_range = valid & (estimates >= lo) & (estimates <= hi)
        achieved = bool(in_range.all())
        if not achieved:
            warnings.warn(
                f"Saved {len(paths)} images, but only {int(in_range.sum())}/{len(estimates)} "
                f"adjacent pairs have estimated overlap {lo:.0%}–{hi:.0%}; "
                f"{int((~valid).sum())} pairs could not be estimated. "
                "Camera motion/parallax may prevent both constraints from being met. "
                "Review frames.csv and consider a slower capture or a shorter video segment.",
                stacklevel=2,
            )
        result = {
            "image_paths": paths,
            "num_images": len(paths),
            "mean_estimated_overlap": float(estimates[valid].mean()) if valid.any() else None,
            "pairs_in_overlap_range": int(in_range.sum()),
            "unknown_pairs": int((~valid).sum()),
            "all_pairs_in_overlap_range": achieved,
            "manifest": output_dir / "frames.csv",
            "start_time": start_time,
            "end_time": end_time,
        }
        print(f"Saved {len(paths)} images to {output_dir}")
        print(f"Pairs in target range: {int(in_range.sum())}/{len(estimates)}")
        return result
    finally:
        cap.release()


In [3]:
# Đổi đường dẫn video và chọn thư mục đầu ra mới/rỗng trước khi chạy.
video_path = Path(r"C:\Users\ASUS\Downloads\IMG_0810.MOV")
output_dir = Path(r"data\raw\capture_frames(1st_60s)")

result = video_to_frames(
    video_path,
    output_dir,
    start_time=0.0,
    end_time=60.0,  # Chỉ phân tích 10 giây đầu
    min_images=80,
    max_images=150,
    target_overlap=0.75,
    overlap_range=(0.70, 0.80),
    count_step=10,  # Dùng 1 để thử mọi số lượng; sẽ chậm hơn.
)
result


Analyzing 0.000–60.000s (1800 frames); trying image counts: [80, 90, 100, 110, 120, 130, 140, 150]


C:\Users\ASUS\AppData\Local\Temp\ipykernel_18496\1808434648.py:5: UserWarning: Saved 80 images, but only 6/79 adjacent pairs have estimated overlap 70%–80%; 0 pairs could not be estimated. Camera motion/parallax may prevent both constraints from being met. Review frames.csv and consider a slower capture or a shorter video segment.
  result = video_to_frames(


Saved 80 images to data\raw\capture_frames(1st_60s)
Pairs in target range: 6/79


{'image_paths': [WindowsPath('data/raw/capture_frames(1st_60s)/frame_0001.jpg'),
  WindowsPath('data/raw/capture_frames(1st_60s)/frame_0002.jpg'),
  WindowsPath('data/raw/capture_frames(1st_60s)/frame_0003.jpg'),
  WindowsPath('data/raw/capture_frames(1st_60s)/frame_0004.jpg'),
  WindowsPath('data/raw/capture_frames(1st_60s)/frame_0005.jpg'),
  WindowsPath('data/raw/capture_frames(1st_60s)/frame_0006.jpg'),
  WindowsPath('data/raw/capture_frames(1st_60s)/frame_0007.jpg'),
  WindowsPath('data/raw/capture_frames(1st_60s)/frame_0008.jpg'),
  WindowsPath('data/raw/capture_frames(1st_60s)/frame_0009.jpg'),
  WindowsPath('data/raw/capture_frames(1st_60s)/frame_0010.jpg'),
  WindowsPath('data/raw/capture_frames(1st_60s)/frame_0011.jpg'),
  WindowsPath('data/raw/capture_frames(1st_60s)/frame_0012.jpg'),
  WindowsPath('data/raw/capture_frames(1st_60s)/frame_0013.jpg'),
  WindowsPath('data/raw/capture_frames(1st_60s)/frame_0014.jpg'),
  WindowsPath('data/raw/capture_frames(1st_60s)/frame_0015.jp